# HECO validation test v.3
---
## Climate condition analysis

this notebook is used to plot rosediagram and map for a characterization of climate condition at the location and date used for the validation test.
-> East mediterranean sea in the date 2021-07-23 to 2021-08-12, accordin to the study conducted about the oil spills that occurred at the Syrian Baniyas Station in the Eastern Mediterranean on August 23, 2021 (DOI: 10.1016/j.marpolbul.2023.115887).

In [1]:
import sys
print(sys.version)
import ipykernel
print(ipykernel.get_connection_file())

3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.4.2)]
/Users/gianfrancodipietro/Library/Jupyter/runtime/kernel-v337c7aecebbb88be07ad4b8d23f1ae974c3e32910.json


In [2]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import pandas as pd
import numpy as np
import os
import sys
import copernicusmarine
import windrose
import warnings
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
import xarray as xr
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import ssl
import reportlab


In [3]:
from reportlab.lib.pagesizes import A4, inch
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import BaseDocTemplate, PageTemplate, Frame, Paragraph, Table, Image, PageBreak
from reportlab.platypus import Image as ResizeImage
from reportlab.lib.utils import ImageReader
from io import BytesIO


In [ ]:
# check dataset downloade

# check yaml input files
import yaml
import os

# prepare directories
output_dir = 'calibration_D_test/geojson'
os.makedirs(output_dir, exist_ok=True)
input_dir = 'calibration_D_test/input_yaml_files'
os.makedirs(input_dir, exist_ok=True)



import yaml
import os
import json

# initialize perturbated variables


# Base configuration structure
base_config = {
    'input': {
        'dataset_file_name': 'med-cmcc-cur-rean-h_1751886654126.nc',
        'lat0': 0,
        'lon0': 0,
        'sim_diffusion_coeff': 0,
        'sim_duration_h': 20.0,
        'sim_particles': 100.0,
        'sim_timedelta_s': 3600.0,
        'spill_release_duration_h': 1.0,
        'time0': '2021-08-29 16:36:07',
        'volume_spilled_m3': 100.0
    }
}

with open('virtual_points_oil_spill-test2/virtual_origin_oil_spill_points_2.geojson', 'r') as f:
    geojson_data = f.read()
    geojson_points = json.loads(geojson_data)

feature_count = len(geojson_points.get("features", []))

# Create the base test file with the initial configuration
file_name = os.path.join(input_dir, "input_test_0.yaml")
with open(file_name, 'w') as yaml_file:
    yaml.dump(base_config, yaml_file, default_flow_style=False, sort_keys=False)

# create a yaml file for each point
num_tests = 10
# TODO: ogni 0.5 m^2/s

for j in range(feature_count):
    point = geojson_points["features"][j]
    lat = point['geometry']['coordinates'][1]
    lon = point['geometry']['coordinates'][0]
    for i in range(num_tests):
        diffusion_coeff = 5.0 + (50.0 - 1.0) * i / max(num_tests - 1, 1) # perturbation of D
        base_config['input']['lat0'] = lat
        base_config['input']['lon0'] = lon
        base_config['input']['sim_diffusion_coeff'] = diffusion_coeff
        file_name = os.path.join(input_dir, f'input_test_point{j}-test_{i}.yaml')
        #print(file_name)
        with open(file_name, 'w') as yaml_file:
            yaml.dump(base_config, yaml_file, default_flow_style=False, sort_keys=False)

print(f'Generated {num_tests * feature_count} yaml files')



Generated 520 yaml files


In [32]:
import heco
import geopandas as gpd
import glob

output_dir = 'calibration_D_test/geojson'
os.makedirs(output_dir, exist_ok=True)

for j in range(feature_count):
    for i in range(num_tests):
        yaml_file_path = os.path.join(input_dir, f'input_test_point{j}-test_{i}.yaml')
        print(f"Running HECO for {i}-{j}")
        output = heco.run(yaml_file_path, 'uo', 'vo')
        gdf = gpd.GeoDataFrame(output, geometry=gpd.points_from_xy(output.lon, output.lat))
        gdf.crs = "EPSG:4326"
        gejson_output_path = os.path.join(output_dir, f'results_point{j}-test_{i}.geojson')
        gdf.to_file(gejson_output_path, driver='GeoJSON')



Running HECO for 0-0
Dataset med-cmcc-cur-rean-h_1751886654126.nc opened
Volume per particle considered: 1.0 m3
discrete spill step 0 , release time 2021-08-29 16:36:07
Running HECO for 1-0
Dataset med-cmcc-cur-rean-h_1751886654126.nc opened
Volume per particle considered: 1.0 m3
discrete spill step 0 , release time 2021-08-29 16:36:07
Running HECO for 2-0
Dataset med-cmcc-cur-rean-h_1751886654126.nc opened
Volume per particle considered: 1.0 m3
discrete spill step 0 , release time 2021-08-29 16:36:07
Running HECO for 3-0
Dataset med-cmcc-cur-rean-h_1751886654126.nc opened
Volume per particle considered: 1.0 m3
discrete spill step 0 , release time 2021-08-29 16:36:07
Running HECO for 4-0
Dataset med-cmcc-cur-rean-h_1751886654126.nc opened
Volume per particle considered: 1.0 m3
discrete spill step 0 , release time 2021-08-29 16:36:07
Running HECO for 5-0
Dataset med-cmcc-cur-rean-h_1751886654126.nc opened
Volume per particle considered: 1.0 m3
discrete spill step 0 , release time 2021-0

KeyboardInterrupt: 